# IPF transcriptome E: CYL + ZCP Scanpy exploration

Workflow: read CYL/ZCP separately, perform per-cohort QC and Scrublet doublet detection, merge retained singlets, then normalize, select HVGs, perform PCA, correct cohort effects with Harmony, cluster and annotate.

# 解释器路径：/home/lijia/jiangyuanpei/miniforge3/envs/allcools/bin/python

In [ ]:
from __future__ import annotations                              # 推迟解析类型注解，使 Path、AnnData 等注解在 Python 3.9 中更安全。

import json                                                     # 用于把最终确认的分析参数写成 JSON 文件。
import tempfile                                                 # 用于创建会自动清理的临时目录。
import zipfile                                                  # 用于读取和解压 ZIP 格式的 10x 表达矩阵。
from pathlib import Path                                        # 提供面向对象的文件路径操作。

import anndata as ad                                            # AnnData 数据结构及多个样本的拼接函数。
import matplotlib.pyplot as plt                                 # Matplotlib 绘图接口，用于创建组合 QC 图。
import numpy as np                                              # 数值计算工具，用于数组和矩阵操作。
import pandas as pd                                             # 表格工具，用于索引、交叉表和 marker 结果整理。
import scanpy as sc                                             # 单细胞分析主库，提供 QC、PCA、UMAP、Leiden 和 marker 分析。
import scanpy.external as sce                                   # Scanpy 外部整合接口，用于调用 Harmony。
import scrublet as scr                                          # 独立 Scrublet 实现，用原始 counts 检测 doublet。

sc.settings.verbosity = 2                                       # 设置日志详细程度为 2，显示关键步骤但避免过多调试信息。
sc.set_figure_params(dpi=100, frameon=False)                    # 设置全局图片分辨率，并隐藏坐标轴外框。
print('scanpy', sc.__version__)                                 # 输出 Scanpy 版本，便于记录和复现环境。
print('harmonypy', '0.0.10')                                    # 输出 Harmony 版本。
print('scrublet', '0.2.3')                                      # 输出 Scrublet 版本。


scanpy 1.9.3
harmonypy 0.0.10
scrublet 0.2.3


In [ ]:
PROJECT_DIR = Path('/home/lijia/luozhixiong/IPF_tissue')                                 # 定义项目根目录，后续输入和输出路径都从这里构建。
INPUT_ZIPS = {                                                                           # 建立 cohort 名称到本地 10x filtered matrix ZIP 的映射。
    'CYL': PROJECT_DIR / 'Data/Matrix/25100718_CYL_E/filtered_feature_bc_matrix.zip',    # CYL 转录组 E 的过滤后表达矩阵。
    'ZCP': PROJECT_DIR / 'Data/Matrix/25100718_ZCP_E/filtered_feature_bc_matrix.zip',    # ZCP 转录组 E 的过滤后表达矩阵。
}
OUTPUT_DIR = PROJECT_DIR / 'Results/Scanpy/E_CYL_ZCP_notebook'                           # 定义参数确认后写出派生分析结果的目录。
FIGURE_DIR = OUTPUT_DIR / 'figures'                                                      # notebook 中显示的图片同步保存到该目录。
FIGURE_DIR.mkdir(parents=True, exist_ok=True)                                            # 绘图前建立输出目录，不等待最终 H5AD 保存开关。
SAVED_FIGURES = []                                                                       # 记录已保存图片，最终导出机器可读 manifest。
RANDOM_SEED = 0                                                                          # 固定随机种子，使降维和聚类结果可复现。

def save_figure(filename, figure=None):                                                  # 保存 Matplotlib/Scanpy 当前图片，同时保留 notebook 内显示。
    path = FIGURE_DIR / filename                                                         # 所有图片集中到固定 notebook 结果目录。
    fig = figure if figure is not None else plt.gcf()                                    # 支持显式 Figure 或 Scanpy 当前 Figure。
    fig.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')                   # 使用统一 300 dpi 白底 PNG 参数。
    SAVED_FIGURES.append(str(path))                                                      # 把实际输出路径加入 manifest。
    print(f'Saved figure: {path}')                                                       # 在 notebook 输出中明确记录文件位置。
    return path                                                                          # 返回路径便于需要时继续使用。

def save_dotplot(plot, filename):                                                        # DotPlot 不是普通 pyplot Figure，单独封装保存。
    path = FIGURE_DIR / filename                                                         # 使用与普通图片相同的目标目录。
    plot.savefig(path, dpi=300, bbox_inches='tight')                                     # 保存完整 dotplot、图例和树状图。
    SAVED_FIGURES.append(str(path))                                                      # 把 DotPlot 输出加入 manifest。
    print(f'Saved figure: {path}')                                                       # 在 notebook 中记录实际路径。
    return path                                                                          # 返回保存路径。

for cohort, path in INPUT_ZIPS.items():                                                  # 逐个检查输入文件是否存在，并打印文件大小。
    assert path.is_file(), f'Missing input for {cohort}: {path}'                         # 文件不存在时立即停止，避免后续出现不明确的读取错误。
    print(cohort, path, f'{path.stat().st_size / 1024**2:.1f} MiB')                      # 以 MiB 为单位打印 cohort、路径和文件大小。

: 

In [ ]:
def read_10x_zip(path: Path, cohort: str) -> ad.AnnData:                                    # 定义读取单个 cohort 的 10x ZIP 并返回 AnnData 的函数。
    with tempfile.TemporaryDirectory(prefix=f'{cohort}_10x_') as tmp:                       # 创建临时目录；离开 with 后目录会自动删除。
        with zipfile.ZipFile(path) as archive:                                              # 以只读方式打开输入 ZIP。
            archive.extractall(tmp)                                                         # 把 ZIP 内容解压到临时目录。
        matrix_dir = Path(tmp) / 'filtered_feature_bc_matrix'                               # 指向 ZIP 内标准的 filtered_feature_bc_matrix 目录。
        sample = sc.read_10x_mtx(matrix_dir, var_names='gene_symbols', make_unique=True)    # 读取 10x 稀疏矩阵，以基因符号为变量名并自动处理重复基因名。
    sample.obs_names = pd.Index([f'{cohort}_{barcode}' for barcode in sample.obs_names])    # 给 barcode 加 cohort 前缀，防止不同样本出现相同细胞 ID。
    sample.obs['cohort'] = cohort                                                           # 在细胞元数据中记录样本来源。
    return sample                                                                           # 返回该 cohort 的 AnnData 对象。

: 

In [ ]:
samples = {cohort: read_10x_zip(path, cohort) for cohort, path in INPUT_ZIPS.items()}                          # 分别读取 CYL 和 ZCP，尚不合并。
for cohort, sample in samples.items():                                                                         # 逐个准备每个 cohort 的原始 counts 与 QC 指标。
    sample.layers['counts'] = sample.X.copy()                                                                  # 保存未经归一化的整数 counts，供 Scrublet 使用。
    sample.var['mt'] = sample.var_names.str.upper().str.startswith('MT-')                                      # 标记线粒体基因。
    sample.var['ribo'] = sample.var_names.str.upper().str.startswith(('RPL', 'RPS'))                           # 标记核糖体蛋白基因。
    sc.pp.calculate_qc_metrics(sample, qc_vars=['mt', 'ribo'], percent_top=None, log1p=False, inplace=True)    # 写入每细胞 QC 指标。
    print(cohort, sample)                                                                                      # 显示该样本的细胞数、基因数和元数据。


: 

## 1. Explore raw counts and choose QC thresholds
Do not finalize thresholds from the S12 notebook automatically. Compare distributions by cohort and record the chosen cutoffs.

In [ ]:
for cohort, sample in samples.items():                          # 分样本查看最高表达基因，避免合并后掩盖 cohort 差异。
    print(f'Highest-expression genes: {cohort}')                # 标明当前图对应的 cohort。
    sc.pl.highest_expr_genes(sample, n_top=20, show=False)      # 绘制该 cohort 占总 counts 比例最高的 20 个基因。
    save_figure(f'{cohort}_highest_expression_genes.png')       # 同步保存该 cohort 的最高表达基因图。
    plt.show()                                                  # 保留 notebook 内嵌显示。


: 

In [ ]:
qc_metrics = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt', 'pct_counts_ribo']                       # 指定用于检查的细胞 QC 指标。
qc_distributions = {}                                                                                        # 建立字典保存每个 cohort 的描述统计。
for cohort, sample in samples.items():                                                                       # 分 cohort 汇总 QC 分布。
    qc_distributions[cohort] = sample.obs[qc_metrics].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])    # 计算分位数与描述统计。
pd.concat(qc_distributions, names=['cohort', 'statistic'])                                                   # 显示每个 cohort 的 QC 分布，作为阈值依据。


: 

In [ ]:
for cohort, sample in samples.items():                                                                    # 对每个 cohort 单独绘图，避免合并分布掩盖差异。
    print(f'QC plots: {cohort}')                                                                          # 标明当前图对应的 cohort。
    fig, axs = plt.subplots(1, len(qc_metrics), figsize=(4 * len(qc_metrics), 4), layout='constrained')   # 为四个 QC 指标创建 Matplotlib 小提琴图画布。
    for ax, metric in zip(axs, qc_metrics):                                                               # 逐个绘制检测基因数、counts、线粒体和核糖体比例。
        values = sample.obs[metric].dropna().to_numpy()                                                   # 取出当前 QC 指标的非缺失数值。
        ax.violinplot(values, showmedians=True)                                                           # 用 Matplotlib 绘制分布并显示中位数，避开 Scanpy/seaborn 兼容问题。
        ax.set_title(metric)                                                                              # 标注当前子图对应的 QC 指标。
        ax.set_xticks([])                                                                                 # 删除无意义的单一样本 x 轴刻度。
    fig.suptitle(f'{cohort}: per-cell QC distributions')                                                  # 标注该整组小提琴图的 cohort。
    save_figure(f'{cohort}_qc_violin_prefilter.png', fig)                                                 # 保存四项 QC 小提琴图。
    plt.show()                                                                                            # 保留 notebook 内嵌显示。
    fig, axs = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')                                  # 创建并排的两张 QC 散点图。
    sc.pl.scatter(sample, x='total_counts', y='pct_counts_mt', ax=axs[0], show=False)                     # 检查总 counts 与线粒体比例的关系。
    sc.pl.scatter(sample, x='total_counts', y='n_genes_by_counts', ax=axs[1], show=False)                 # 检查总 counts 与检测基因数的关系。
    save_figure(f'{cohort}_qc_scatter_prefilter.png', fig)                                                # 保存两项 QC 关系散点图。
    plt.show()                                                                                            # 保留 notebook 内嵌显示。


: 

### Candidate thresholds
The values below are starting points from the template, not confirmed project parameters. Adjust them after inspecting the plots and per-cohort pass counts.

In [ ]:
MIN_GENES = 200                                                                               # 每个细胞至少检测到 200 个基因，用于排除低复杂度液滴。
MAX_GENES = 6000                                                                              # 每个细胞最多检测到 6000 个基因，用于减少潜在 doublet。
MIN_COUNTS = 500                                                                              # 每个细胞至少保留 500 个总 counts。
MAX_MT_PERCENT = 5.0                                                                          # 线粒体 counts 比例上限；应依据各 cohort 的 QC 分布人工确认。
MIN_CELLS_PER_GENE = 3                                                                        # 仅保留至少在 3 个通过细胞 QC 的细胞中表达的基因。
DOUBLET_RATE_PER_1000 = 0.004                                                                 # 每回收 1,000 个细胞对应的预期 doublet 比例。

qc_counts = {}                                                                                # 保存每个 cohort 的基础 QC 通过数量。
for cohort, sample in samples.items():                                                        # 为每个 cohort 独立应用同一组候选基础 QC 阈值。
    pass_qc = (                                                                               # 同时满足以下四个条件的细胞才通过基础 QC。
        (sample.obs['n_genes_by_counts'] >= MIN_GENES)                                        # 检测基因数不低于下限。
        & (sample.obs['n_genes_by_counts'] < MAX_GENES)                                       # 检测基因数低于上限。
        & (sample.obs['total_counts'] >= MIN_COUNTS)                                          # 总 counts 不低于下限。
        & (sample.obs['pct_counts_mt'] < MAX_MT_PERCENT)                                      # 线粒体比例低于设定上限。
    )
    sample.obs['pass_basic_qc'] = pass_qc                                                     # 把基础 QC 结果写入该样本的细胞元数据。
    qc_counts[cohort] = {'input_cells': sample.n_obs, 'pass_basic_qc': int(pass_qc.sum())}    # 记录过滤前后细胞数。
pd.DataFrame(qc_counts).T                                                                     # 显示每个 cohort 的基础 QC 保留数量。


: 

In [ ]:
samples_qc = {}                                                                                   # 保存通过基础 QC 且去除 Scrublet doublet 的每个 cohort。
qc_summary_rows = []                                                                              # 收集完整的细胞保留数量，供最终汇总。
for cohort, sample in samples.items():                                                            # 独立处理每个 cohort，禁止在 Scrublet 前合并。
    sample_qc = sample[sample.obs['pass_basic_qc']].copy()                                        # 仅保留基础 QC 通过的细胞，原样本保持不变。
    sc.pp.filter_genes(sample_qc, min_cells=MIN_CELLS_PER_GENE)                                   # 在该 cohort 内去除低频基因。
    if sample_qc.n_obs < 100:                                                                     # 防止细胞过少时 Scrublet 的邻域估计不稳定。
        raise RuntimeError(f'{cohort} has fewer than 100 cells after basic QC')                   # 提供明确错误而非产生不可靠 doublet 调用。
    n_cells = sample_qc.n_obs                                                                     # 记录实际进入 Scrublet 的该 cohort 细胞数。
    expected_doublet_rate = DOUBLET_RATE_PER_1000 * n_cells / 1000                                # 按 0.004 × n_cells / 1000 动态计算该 cohort 的先验 doublet 率。
    scrub = scr.Scrublet(                                                                         # 用当前 cohort 的原始整数 counts 初始化 Scrublet。
        sample_qc.layers['counts'],                                                               # 传入过滤后的原始 counts，而非归一化或 log 数据。
        expected_doublet_rate=expected_doublet_rate,                                              # 传入按该 cohort 细胞数动态计算的先验 doublet 率。
        random_state=RANDOM_SEED,                                                                 # 固定模拟随机种子，确保可复现。
    )
    doublet_scores, predicted_doublets = scrub.scrub_doublets(                                    # 计算每个细胞的 doublet 分数和预测标签。
        n_prin_comps=30,                                                                          # 在 30 个主成分空间中区分观测与模拟 doublet。
        use_approx_neighbors=False,                                                               # 对本数据规模使用精确近邻，避免近似算法带来额外随机性。
        verbose=True,                                                                             # 在运行时输出 Scrublet 的关键进度信息。
    )
    sample_qc.obs['doublet_score'] = doublet_scores                                               # 写入连续 doublet 分数，供后续审核。
    sample_qc.obs['predicted_doublet'] = predicted_doublets                                       # 写入 Scrublet 自动阈值给出的预测标签。
    scrub.plot_histogram()                                                                        # 绘制 observed/simulated doublet 分数分布，必须人工检查阈值合理性。
    save_figure(f'{cohort}_scrublet_histogram.png')                                               # 保存该 cohort 的 Scrublet 分数直方图。
    plt.show()                                                                                    # 保留 notebook 内嵌显示。
    retained = sample_qc[~sample_qc.obs['predicted_doublet']].copy()                              # 从下游分析中排除预测 doublet，并保留 singlet。
    samples_qc[cohort] = retained                                                                 # 保存该 cohort 的 singlet AnnData。
    qc_summary_rows.append({                                                                      # 记录该 cohort 在每个阶段的细胞数。
        'cohort': cohort,                                                                         # 记录 cohort 名称。
        'input_cells': sample.n_obs,                                                              # 记录原始输入细胞数。
        'pass_basic_qc': int(sample.obs['pass_basic_qc'].sum()),                                  # 记录基础 QC 后的细胞数。
        'expected_doublet_rate': expected_doublet_rate,                                           # 记录本 cohort 实际传给 Scrublet 的先验 doublet 率。
        'predicted_doublets': int(sample_qc.obs['predicted_doublet'].sum()),                      # 记录 Scrublet 预测为 doublet 的细胞数。
        'retained_singlets': retained.n_obs,                                                      # 记录最终进入合并的 singlet 数。
    })

qc_summary = pd.DataFrame(qc_summary_rows).set_index('cohort')                                    # 整理并显示两组样本的 QC/Scrublet 汇总。
display(qc_summary)                                                                               # 在 notebook 中渲染汇总表。
adata_qc = ad.concat(list(samples_qc.values()), join='outer', merge='same', index_unique=None)    # 合并各 cohort 已完成 QC 和 doublet 去除的 singlet。
adata_qc.obs['cohort'] = adata_qc.obs['cohort'].astype('category')                                # 将 cohort 设为分类变量，便于下游批次感知 HVG。
assert not adata_qc.obs_names.duplicated().any()                                                  # 再次确认合并后细胞 ID 唯一。
print('merged singlets:', adata_qc.shape)                                                         # 显示最终进入归一化的细胞数和基因数。


: 

## 2. Normalize, select HVGs and inspect PCA
HVGs are selected with `cohort` as the batch key. Cohort correction is not applied automatically because CYL/ZCP may encode biology as well as technical effects.

In [ ]:
N_HVG = 2000                                                                           # 计划选择 2000 个高变基因用于降维和聚类。
sc.pp.normalize_total(adata_qc, target_sum=1e4)                                        # 在已合并的 singlet 上将每个细胞总表达量归一化到 10,000。
sc.pp.log1p(adata_qc)                                                                  # 对归一化表达值执行 log(1+x) 转换，压缩高表达值范围。
adata_qc.raw = adata_qc                                                                # 保存全基因归一化表达矩阵，供 marker 检验和绘图使用。
sc.pp.highly_variable_genes(                                                           # 在已合并的 singlet 中选择高变基因。
    adata_qc, n_top_genes=N_HVG, batch_key='cohort', flavor='seurat'                   # 按 cohort 分批计算后合并；seurat 方法不依赖 scikit-misc。
)
sc.pl.highly_variable_genes(adata_qc, show=False)                                      # 绘制高变基因选择诊断图。
save_figure('highly_variable_genes.png')                                               # 保存高变基因选择诊断图。
plt.show()                                                                             # 保留 notebook 内嵌显示。
adata_qc.var[['highly_variable', 'highly_variable_nbatches']].value_counts().head()    # 统计高变标记及其在多少个 cohort 中被选中的组合。


: 

In [ ]:
REGRESS_COVARIATES = False                                                             # 默认不回归总 counts 和线粒体比例；确认其影响后再改为 True。
adata_work = adata_qc[:, adata_qc.var['highly_variable']].copy()                       # 只保留高变基因并复制为后续降维使用的工作对象。
if REGRESS_COVARIATES:                                                                 # 仅在明确启用时回归两个技术协变量。
    sc.pp.regress_out(adata_work, ['total_counts', 'pct_counts_mt'])                   # 从每个基因的表达中回归总 counts 和线粒体比例。
sc.pp.scale(adata_work, max_value=10)                                                  # 对每个基因做中心化和标准化，并把极端标准化值截断为 10。
sc.tl.pca(adata_work, n_comps=50, svd_solver='arpack', random_state=RANDOM_SEED)       # 使用 ARPACK 计算前 50 个主成分，并固定随机种子。
sc.pl.pca_variance_ratio(adata_work, n_pcs=50, log=True, show=False)                   # 绘制主成分解释方差比例，辅助决定实际使用的 PC 数。
save_figure('pca_variance_ratio.png')                                                  # 保存 PCA 解释方差图。
plt.show()                                                                             # 保留 notebook 内嵌显示。
sc.pl.pca(adata_work, color='cohort', show=False)                                      # 显示 Harmony 前的原始 PCA，作为批次校正效果的对照。
save_figure('pca_before_harmony_by_cohort.png')                                        # 保存 Harmony 前 PCA 图。
plt.show()                                                                             # 保留 notebook 内嵌显示。
HARMONY_BATCH_KEY = 'cohort'                                                           # 指定需要校正的批次标签；此处为 CYL/ZCP。
HARMONY_BASIS = 'X_pca_harmony'                                                        # 指定 Harmony 校正后 PCA 坐标在 obsm 中的名称。
N_HARMONY_CLUSTERS = min(100, max(2, int(np.round(adata_work.n_obs / 30))))            # 显式设置 Harmony 初始聚类数，避免小数据边界问题。
HARMONY_SIGMA = np.repeat(0.1, N_HARMONY_CLUSTERS)                                     # 为每个 Harmony 初始聚类提供一个 sigma 值。
sce.pp.harmony_integrate(                                                              # 在原始 X_pca 上执行 Harmony，并保留原始 PCA 不被覆盖。
    adata_work, key=HARMONY_BATCH_KEY, basis='X_pca', adjusted_basis=HARMONY_BASIS,    # 指定输入 PCA、批次列和输出嵌入名称。
    nclust=N_HARMONY_CLUSTERS, sigma=HARMONY_SIGMA,                                    # 显式传入初始聚类数及对应 sigma 数组。
    max_iter_harmony=20, random_state=RANDOM_SEED,                                     # 限制迭代次数并固定 Harmony 的随机种子。
)
assert HARMONY_BASIS in adata_work.obsm                                                # 确认校正后的低维表示已成功生成。


: 

## 3. Neighbours, UMAP and clustering
Inspect cohort separation before deciding whether a batch-corrected representation is justified. Try parameter alternatives in copied cells rather than overwriting the first result.

In [ ]:
N_PCS = 30                                                                                                                        # 去批次前后均使用前 30 个 PCA/Harmony 成分构建邻接图。
N_NEIGHBORS = 15                                                                                                                  # 去批次前后均使用每细胞 15 个近邻，确保对照只改变表示。
SAMPLE_KEY = 'sample'                                                                                                             # 定义最终 sample UMAP 使用的元数据列。
GROUP_KEY = 'group'                                                                                                               # 预留生物学 group 列；当前数据尚未提供该信息。
adata_work.obs[SAMPLE_KEY] = adata_work.obs['cohort'].astype(str).astype('category')                                              # 当前 cohort 即 CYL/ZCP 样本名，复制为明确的 sample 元数据。
BEFORE_NEIGHBORS_KEY = 'neighbors_before_harmony'                                                                                 # 定义原始 PCA 邻接图名称，避免覆盖 Harmony 邻接图。
BEFORE_UMAP_BASIS = 'X_umap_before_harmony'                                                                                       # 定义去批次前 UMAP 坐标名称。
AFTER_UMAP_BASIS = 'X_umap_after_harmony'                                                                                         # 定义去批次后 UMAP 坐标名称。
sc.pp.neighbors(                                                                                                                  # 使用未经 Harmony 校正的原始 PCA 构建对照邻接图。
    adata_work, n_neighbors=N_NEIGHBORS, use_rep='X_pca', n_pcs=N_PCS,                                                            # 保持近邻数和 PC 数与 Harmony 后完全一致。
    random_state=RANDOM_SEED, key_added=BEFORE_NEIGHBORS_KEY,                                                                     # 固定随机种子并把对照图保存到独立 key。
)
sc.tl.umap(adata_work, neighbors_key=BEFORE_NEIGHBORS_KEY, random_state=RANDOM_SEED)                                              # 从原始 PCA 邻接图计算去批次前 UMAP。
adata_work.obsm[BEFORE_UMAP_BASIS] = adata_work.obsm['X_umap'].copy()                                                             # 保存去批次前坐标，防止下一次 UMAP 覆盖。
sc.pp.neighbors(                                                                                                                  # 使用 Harmony 校正后的 PCA 构建正式邻接图。
    adata_work, n_neighbors=N_NEIGHBORS, use_rep=HARMONY_BASIS, n_pcs=N_PCS, random_state=RANDOM_SEED,                            # 除输入表示外保持其他参数与对照一致。
)
sc.tl.umap(adata_work, random_state=RANDOM_SEED)                                                                                  # 从默认 Harmony 邻接图计算正式 UMAP。
adata_work.obsm[AFTER_UMAP_BASIS] = adata_work.obsm['X_umap'].copy()                                                              # 保存去批次后坐标；默认 X_umap 同时保持为正式坐标。
fig, axs = plt.subplots(1, 2, figsize=(13, 5), layout='constrained')                                                              # 创建去批次前后的并排 sample 对照图。
sc.pl.embedding(                                                                                                                  # 在原始 PCA UMAP 上按 sample 着色。
    adata_work, basis='umap_before_harmony', color=SAMPLE_KEY, ax=axs[0], show=False,                                             # 左图展示 Harmony 前的 CYL/ZCP 分布。
    size=10, alpha=0.75, frameon=False, title='Before Harmony', palette=['#0072B2', '#E69F00'],                                   # 使用与正式 sample 图一致的配色和点参数。
)
sc.pl.embedding(                                                                                                                  # 在 Harmony UMAP 上按 sample 着色。
    adata_work, basis='umap_after_harmony', color=SAMPLE_KEY, ax=axs[1], show=False,                                              # 右图展示 Harmony 后的 CYL/ZCP 分布。
    size=10, alpha=0.75, frameon=False, title='After Harmony', palette=['#0072B2', '#E69F00'],                                    # 与左图保持完全相同的视觉参数。
)
save_figure('umap_before_after_harmony_by_sample.png', fig)                                                                        # 保存去批次前后 sample 对照图。
plt.show()                                                                                                                         # 显示去批次前后对照图。
sc.pl.embedding(adata_work, basis='umap_after_harmony', color=['total_counts', 'pct_counts_mt'], frameon=False, show=False)        # 在正式 UMAP 上检查 QC 指标是否仍主导结构。
save_figure('umap_harmony_qc.png')                                                                                                 # 保存 Harmony UMAP 上的 QC 指标图。
plt.show()                                                                                                                         # 保留 notebook 内嵌显示。


: 

In [ ]:
LEIDEN_RESOLUTION = 0.8                                                                                                   # 设置 Leiden 聚类分辨率；数值越高通常得到越多 cluster。
sc.tl.leiden(adata_work, resolution=LEIDEN_RESOLUTION, random_state=RANDOM_SEED)                                          # 在前一步的邻接图上执行 Leiden 社区发现并写入 obs['leiden']。
sc.pl.embedding(adata_work, basis='umap_after_harmony', color='leiden', legend_loc='on data', frameon=False, show=False)  # 仅在正式 Harmony UMAP 上显示数字 cluster 标签。
save_figure('umap_leiden.png')                                                                                            # 保存 Leiden UMAP。
plt.show()                                                                                                                # 保留 notebook 内嵌显示。
pd.crosstab(adata_work.obs['leiden'], adata_work.obs[SAMPLE_KEY], normalize='index')                                      # 计算每个 cluster 内 CYL/ZCP sample 比例，识别样本特异 cluster。

: 

## 4. Cluster markers and manual annotation
Do not reuse the S12 cluster-number mapping. Cluster IDs change with filtering, samples and resolution. Review markers and canonical lung markers before filling `cluster_to_cell_type`.

In [ ]:
sc.tl.rank_genes_groups(adata_work, 'leiden', method='wilcoxon', use_raw=True)    # 使用 Wilcoxon 秩和检验比较各 Leiden cluster，并从 raw 全基因矩阵寻找 marker。
sc.pl.rank_genes_groups(adata_work, n_genes=20, sharey=False, show=False)         # 绘制每个 cluster 排名前 20 的差异表达基因。
save_figure('leiden_top20_ranked_markers.png')                                    # 保存所有 Leiden cluster 的 Top20 marker 图。
plt.show()                                                                        # 保留 notebook 内嵌显示。
marker_names = pd.DataFrame(adata_work.uns['rank_genes_groups']['names'])         # 把 Scanpy 保存的 marker 基因名结构转换为便于阅读的 DataFrame。
marker_names.head(25)                                                             # 显示每个 cluster 排名前 15 的候选 marker。

: 

In [ ]:
marker_genes = {                                                                                                           # 用互相独立的 marker 组合验证谱系和容易混淆的亚型。
    'Pan_epithelial': ['EPCAM', 'KRT8', 'KRT18', 'KRT19'],                                                                 # 泛上皮 marker，用于判断 cycling/未定群的谱系来源。
    'AT2': ['SFTPC', 'SFTPB', 'SFTPA1', 'SFTPA2', 'ABCA3', 'LPCAT1'],                                                      # AT2 的表面活性物质合成与板层小体 marker。
    'AT1': ['AGER', 'CAV1', 'CAV2', 'PDPN', 'HOPX', 'EMP2', 'AQP5'],                                                       # AT1 的薄层肺泡上皮 marker。
    'Secretory': ['SCGB1A1', 'SCGB3A1', 'SCGB3A2', 'BPIFB1', 'MUC4', 'WFDC2', 'SLPI'],                                     # 分泌型气道上皮 marker，用于区分 AT2-like 群。
    'Basal': ['KRT5', 'KRT14', 'KRT15', 'KRT17', 'TP63', 'MIR205HG'],                                                      # 基底细胞 marker。
    'Ciliated': ['FOXJ1', 'PIFO', 'TPPP3', 'DNAH11', 'CFAP46', 'HYDIN'],                                                   # 运动纤毛及轴丝相关 marker。
    'Macrophage': ['LST1', 'TYROBP', 'FCER1G', 'CD68', 'C1QA', 'C1QB', 'C1QC', 'MRC1', 'CD163', 'PPARG'],                  # 肺巨噬细胞/髓系 marker。
    'Fibroblast': ['COL1A1', 'COL1A2', 'COL3A1', 'DCN', 'LUM', 'PDGFRA', 'COL6A3'],                                        # 成纤维细胞及细胞外基质 marker。
    'Pan_endothelial': ['PECAM1', 'VWF', 'KDR', 'EMCN', 'ENG', 'ESAM', 'RAMP2', 'PLVAP'],                                  # 血管内皮共有 marker。
    'Capillary_endothelial': ['CA4', 'RGCC', 'EMCN', 'GPIHBP1', 'BTNL9', 'EDNRB', 'EPAS1'],                                # 肺毛细血管内皮 marker。
    'Lymphatic_endothelial': ['PROX1', 'PDPN', 'LYVE1', 'FLT4', 'CCL21', 'MMRN1', 'RELN'],                                 # 淋巴内皮 marker，用于验证当前 cluster 15。
    'Smooth_muscle_mural': ['ACTA2', 'TAGLN', 'MYH11', 'LMOD1', 'CNN1', 'CARMN', 'PDGFRB', 'RGS5'],                        # 平滑肌/周细胞谱系 marker。
    'T_cell': ['CD3D', 'CD3E', 'TRAC', 'BCL11B', 'ITK', 'CD247', 'IL7R'],                                                  # T 细胞受体及 T 谱系 marker。
    'Mast': ['KIT', 'CPA3', 'TPSAB1', 'TPSB2', 'HDC', 'MS4A2', 'HPGDS'],                                                   # 肥大细胞受体、蛋白酶和组胺合成 marker。
    'Cycling': ['MKI67', 'TOP2A', 'UBE2C', 'CENPF', 'BIRC5', 'RRM2', 'ANLN', 'ECT2', 'DIAPH3'],                            # 完整细胞周期 marker，用于区分 cluster 14 与 cluster 16。
}
dotplot_markers = {                                                                                                        # 正式点图每个细胞类型只保留 4 个代表性 marker。
    'AT2': ['SFTPC', 'ABCA3', 'NAPSA', 'LPCAT1'],                                                                          # AT2 marker。
    'Secretory epithelial': ['NEDD4L', 'SFTA3', 'SCNN1B', 'GPRC5A'],                                                       # 分泌型上皮 marker。
    'Fibroblasts': ['COL1A1', 'COL1A2', 'DCN', 'COL3A1'],                                                                  # 成纤维细胞 marker。
    'Ciliated cells': ['FOXJ1', 'DNAH11', 'PIFO', 'CFAP46'],                                                               # 纤毛细胞 marker。
    'Secretory / mucous epithelial': ['BPIFB1', 'MUC4', 'WFDC2', 'TMC5'],                                                  # 分泌/黏液上皮 marker。
    'Macrophages': ['LST1', 'C1QA', 'MRC1', 'CD163'],                                                                      # 巨噬细胞 marker。
    'AT1-like': ['AGER', 'CAV1', 'HOPX', 'AQP5'],                                                                          # AT1 marker。
    'Endothelial cells': ['PECAM1', 'VWF', 'KDR', 'EMCN'],                                                                 # 血管内皮 marker。
    'Basal cells': ['KRT5', 'KRT15', 'TP63', 'KRT17'],                                                                     # 基底细胞 marker。
    'Smooth muscle / mural cells': ['ACTA2', 'MYH11', 'CARMN', 'PDGFRB'],                                                  # 平滑肌/壁细胞 marker。
    'MT-high AT2-like': ['MT-CO1', 'MT-ND1', 'MT-ND4', 'MT-CYB'],                                                          # 线粒体高表达 AT2-like marker。
    'T cells': ['CD3D', 'CD3E', 'TRAC', 'BCL11B'],                                                                         # T 细胞 marker。
    'Cycling cells': ['MKI67', 'TOP2A', 'RRM2', 'ANLN'],                                                                   # cycling marker。
    'Lymphatic endothelial cells': ['PROX1', 'FLT4', 'CCL21', 'LYVE1'],                                                    # 淋巴内皮 marker。
    'NA': ['EPCAM', 'KRT8', 'COL11A1', 'CEMIP'],                                                                           # 合并 cluster 16/17 后同时展示上皮与基质线索。
}                                                                                                                          # 顶部分组名称与合并后的 cell_type 完全一致。
cell_type_order = list(dotplot_markers)                                                                                    # 以同一顺序初始化纵轴细胞类型和顶部 marker 分组。
epithelial_clusters = ['1', '4', '6', '10', '12', '14', '16']                                                              # 集中审核上皮 compartment。
rare_clusters = ['15', '17']                                                                                               # 单独审核两个稀有群。

: 

In [ ]:
cluster_to_cell_type = {                                                                                               # 本映射仅对应当前审核的 18-cluster marker 结果。
    '0': 'AT2',                                                                                                        # SFTPC/SFTPB/ABCA3/LPCAT1 支持 AT2。
    '1': 'Secretory epithelial',                                                                                       # NEDD4L/SFTPB/SFTA3 与上皮区定位支持分泌型上皮。
    '2': 'Fibroblasts',                                                                                                # COL1A2/COL3A1/COL5A1/COL6A3/PDGFRA 支持成纤维细胞。
    '3': 'Ciliated cells',                                                                                             # CFAP/DNAH/HYDIN 等运动纤毛基因强烈富集。
    '4': 'Secretory / mucous epithelial',                                                                              # BPIFB1/MUC4/ERN2/TMC5 支持分泌/黏液上皮。
    '5': 'Macrophages',                                                                                                # CD163/MRC1/CTSB/FCER1G 支持巨噬细胞。
    '6': 'AT1-like',                                                                                                   # CAV1/HOPX/CAV2 上升但 AGER/PDPN/AQP5 不完整，且保留明显 AT2 marker。
    '7': 'Endothelial cells',                                                                                          # EPAS1/PECAM1/VWF/BTNL9 支持内皮。
    '8': 'Endothelial cells',                                                                                          # VWF/PTPRB/PECAM1/EPAS1 支持内皮。
    '9': 'Macrophages',                                                                                                # PPARG/MRC1/CD163/MSR1 支持另一巨噬细胞状态。
    '10': 'Basal cells',                                                                                               # EGFR/KRT15/COL7A1/TP63/KRT5 支持基底细胞。
    '11': 'Smooth muscle / mural cells',                                                                               # MYH11/LMOD1/CARMN/PDGFRB/COL4A1 支持平滑肌/血管壁细胞。
    '12': 'MT-high AT2-like',                                                                                          # MT genes 与 SFTPC/SFTPA2/SFTPB/ABCA3 支持受压 AT2-like 状态。
    '13': 'T cells',                                                                                                   # PTPRC/CD247/BCL11B/ITK/DOCK2 支持 T 细胞。
    '14': 'Cycling cells',                                                                                             # DIAPH3/FANCI/MELK/RRM2/ECT2/ANLN 支持完整 cycling 程序；谱系待专项面板确认。
    '15': 'Lymphatic endothelial cells',                                                                               # PROX1/FLT4/LYVE1/CCL21/MMRN1/RELN 联合支持淋巴内皮。
    '16': 'NA',                                                                                                        # 50-cell、CYL 偏倚的未定上皮样小群，一级注释按用户指定设为 NA。
    '17': 'NA',                                                                                                        # 20-cell、ZCP 偏倚的未定基质样小群，一级注释按用户指定设为 NA。
}
observed_clusters = set(adata_work.obs['leiden'].astype(str).unique())                                                    # 读取本次实际产生的 Leiden 编号。
expected_clusters = set(cluster_to_cell_type)                                                                             # 读取人工审核过的 18 个编号。
if observed_clusters != expected_clusters:                                                                                # 聚类编号一旦变化就禁止静默套用旧标签。
    missing = sorted(expected_clusters - observed_clusters, key=int)                                                      # 列出本次缺失的旧 cluster。
    new = sorted(observed_clusters - expected_clusters, key=int)                                                          # 列出本次新增且尚未审核的 cluster。
    raise RuntimeError(f'Cluster IDs changed; missing={missing}, new={new}. Recheck markers before annotation.')          # 要求重新检查 marker 后更新映射。
adata_work.obs['cell_type'] = pd.Categorical(adata_work.obs['leiden'].astype(str).map(cluster_to_cell_type),              # 把 18 个 cluster 合并成审核后的细胞类型。
                                                 categories=cell_type_order, ordered=True)                                # 纵轴与顶部 marker 分组共用同一初始顺序。
def draw_marker_dotplot(adata, panels):                                                                                   # 统一生成示例所示的分组、树状图和红色色阶点图。
    cell_types = list(adata.obs['cell_type'].cat.categories)                                                              # 读取当前子集实际保留的合并细胞类型顺序。
    if set(cell_types) != set(panels):                                                                                    # 检查纵轴类型和顶部 marker 分组能否一一对应。
        raise ValueError(f'cell_type rows and marker groups differ: {cell_types} versus {list(panels)}')                  # 不一致时停止，防止画出错位点图。
    panels = {label: [gene for gene in panels[label] if gene in adata.raw.var_names] for label in cell_types}             # 按纵轴顺序排列 marker，并删除数据中不存在的基因。
    genes = [gene for group in panels.values() for gene in group]                                                         # 按顶部细胞类型分组顺序展开 marker。
    sc.tl.dendrogram(adata, groupby='cell_type', var_names=genes, use_raw=True, cor_method='pearson',                     # 用展示基因的平均表达计算合并细胞类型聚类。
                     linkage_method='complete')                                                                           # 使用 complete linkage 生成右侧 dendrogram。
    plot = sc.pl.dotplot(adata, panels, groupby='cell_type', use_raw=True, dendrogram=True,                               # 纵轴和顶部分组均为合并后的 cell type。
                         var_group_rotation=90, figsize=(max(12, 0.32 * len(genes) + 5),                                  # 顶部分组名纵向显示并按基因数自动调整宽度。
                                                            max(5, 0.48 * adata.obs['cell_type'].nunique() + 3)),         # 按合并细胞类型数自动调整高度。
                         show=False, return_fig=True)                                                                     # 返回 DotPlot 对象以继续统一美化。
    plot.style(cmap='Reds', dot_min=0, dot_max=0.6, smallest_dot=0, largest_dot=180,                                      # 红色表示平均 log-normalized 表达，点面积表示阳性比例。
               dot_edge_color='#888888', dot_edge_lw=0.5, size_exponent=1.5, grid=False).legend(                          # 添加灰色点边框并关闭网格。
        size_title='Fraction of cells\nin group (%)', colorbar_title='Mean expression\nin group', width=1.8)              # 使用与示例一致的两个图例。
    return plot                                                                                                           # 返回已完成样式设置的图对象。
global_dotplot = draw_marker_dotplot(adata_work, dotplot_markers)                                                         # 先合并同类型 cluster，再绘制正式 marker 点图。
save_dotplot(global_dotplot, 'annotation_marker_dotplot.png')                                                             # 保存全局 cell-type marker 点图。
global_dotplot.show()                                                                                                     # 在 notebook 中显示全局点图。
epithelial_groups = ['Secretory epithelial', 'Secretory / mucous epithelial', 'AT1-like', 'Basal cells',                  # 选择专项审核的上皮相关合并类型。
                     'MT-high AT2-like', 'Cycling cells', 'NA']                                                           # 加入 MT-high、cycling 与 NA 类型。
epithelial_subset = adata_work[adata_work.obs['leiden'].astype(str).isin(epithelial_clusters)].copy()                     # 提取上皮相关 cluster。
epithelial_subset.obs['cell_type'] = epithelial_subset.obs['cell_type'].cat.remove_unused_categories()                    # 删除子集中未出现的非上皮类别。
epithelial_dotplot = draw_marker_dotplot(epithelial_subset, {k: dotplot_markers[k] for k in epithelial_groups})           # 生成上皮专项点图。
save_dotplot(epithelial_dotplot, 'epithelial_focus_marker_dotplot.png')                                                   # 保存上皮专项点图。
epithelial_dotplot.show()                                                                                                 # 在 notebook 中显示上皮专项点图。
rare_groups = ['Lymphatic endothelial cells', 'NA']                                                                       # 顶部分组与两个稀有细胞类型逐一对应。
rare_subset = adata_work[adata_work.obs['leiden'].astype(str).isin(rare_clusters)].copy()                                 # 提取 cluster 15 和 17。
rare_subset.obs['cell_type'] = rare_subset.obs['cell_type'].cat.remove_unused_categories()                                # 删除子集中未出现的其他细胞类型。
rare_dotplot = draw_marker_dotplot(rare_subset, {k: dotplot_markers[k] for k in rare_groups})                             # 生成稀有群专项点图。
save_dotplot(rare_dotplot, 'rare_cluster_marker_dotplot.png')                                                             # 保存稀有群专项点图。
rare_dotplot.show()                                                                                                       # 在 notebook 中显示稀有群专项点图。
rare_qc = rare_subset.obs.groupby('leiden', observed=True).agg(                                                           # 汇总稀有群的细胞数、样本偏倚和 QC/doublet 状态。
    cells=('cohort', 'size'), median_genes=('n_genes_by_counts', 'median'), median_counts=('total_counts', 'median'),     # 记录细胞数、检测基因数和总 counts 中位数。
    median_pct_mt=('pct_counts_mt', 'median'), median_doublet_score=('doublet_score', 'median'),                          # 记录线粒体比例和 Scrublet 分数中位数。
)                                                                                                                         # 完成按 Leiden 的稀有群 QC 汇总。
rare_qc.join(pd.crosstab(rare_subset.obs['leiden'], rare_subset.obs['cohort'], normalize='index'))                        # 显示 cluster 15/17 的 QC 与样本构成。
sc.pl.embedding(                                                                                                          # 第一张正式单图：按人工 cell type 着色。
    adata_work, basis='umap_after_harmony', color='cell_type', legend_loc='right margin',                                 # 明确使用 Harmony 后坐标并把长标签移到图外。
    size=10, alpha=0.85, frameon=False, title='Cell type', show=False,                                                    # 使用较小半透明散点并去除坐标框。
)
save_figure('umap_cell_type.png')                                                                                         # 保存最终 cell-type UMAP。
plt.show()                                                                                                                # 保留 notebook 内嵌显示。
sc.pl.embedding(                                                                                                          # 第二张正式单图：按 sample 着色。
    adata_work, basis='umap_after_harmony', color=SAMPLE_KEY, legend_loc='right margin',                                  # 当前 sample 为 CYL/ZCP，不再用含义模糊的 cohort 图标题。
    size=10, alpha=0.75, frameon=False, title='Sample', palette=['#0072B2', '#E69F00'], show=False,                       # 使用色盲友好的蓝橙配色。
)
save_figure('umap_sample.png')                                                                                            # 保存 sample UMAP。
plt.show()                                                                                                                # 保留 notebook 内嵌显示。
if GROUP_KEY in adata_work.obs and adata_work.obs[GROUP_KEY].notna().any():                                               # 仅在真实 group 元数据存在时绘制第三张正式单图。
    sc.pl.embedding(                                                                                                      # 第三张正式单图：按生物学 group 着色。
        adata_work, basis='umap_after_harmony', color=GROUP_KEY, legend_loc='right margin',                               # 使用 Harmony 后坐标，图例放在图外。
        size=10, alpha=0.75, frameon=False, title='Group', show=False,                                                    # group 配色由 Scanpy 根据实际类别生成。
    )
    save_figure('umap_group.png')                                                                                         # 仅在 group 元数据存在时保存 group UMAP。
    plt.show()                                                                                                            # 保留 notebook 内嵌显示。
else:                                                                                                                     # 当前 CYL/ZCP 数据尚未提供 group 元数据。
    print("Group UMAP skipped: obs['group'] is not available; add verified group metadata before plotting.")              # 明确记录跳过原因，不把 sample 伪装成 group。

: 

## 5. Save only after the analysis choices are confirmed
Set `ANALYSIS_CONFIRMED = True` only after documenting final thresholds, PCA/neighbour settings, cluster resolution and cell-type mapping.

In [ ]:
ANALYSIS_CONFIRMED = True                                                                       # 安全开关：只有人工审核全部参数和注释后才可改为 True。
if not ANALYSIS_CONFIRMED:                                                                      # 未确认时主动停止，防止把探索性结果误写成正式输出。
    raise RuntimeError('Review and confirm notebook parameters before saving final outputs')    # 抛出清晰错误并终止本单元。

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)                                                   # 创建输出目录及缺失的父目录；目录已存在时不报错。
adata_work.uns['notebook_parameters'] = {                                                       # 把本次确认的关键参数写入 AnnData.uns，便于结果追溯。
    'min_genes': MIN_GENES, 'max_genes': MAX_GENES, 'min_counts': MIN_COUNTS,                   # 记录细胞的基因数和总 counts 阈值。
    'max_mt_percent': MAX_MT_PERCENT, 'min_cells_per_gene': MIN_CELLS_PER_GENE,                 # 记录线粒体比例阈值和基因最低检出细胞数。
    'n_hvg': N_HVG, 'regress_covariates': REGRESS_COVARIATES,                                   # 记录高变基因数及是否回归技术协变量。
    'doublet_rate_per_1000': DOUBLET_RATE_PER_1000,                                             # 记录 Scrublet 动态先验 doublet 率公式中的系数。
    'n_pcs': N_PCS, 'n_neighbors': N_NEIGHBORS,                                                 # 记录去批次前后邻接图共同使用的 PC 数和近邻数。
    'harmony_batch_key': HARMONY_BATCH_KEY, 'harmony_basis': HARMONY_BASIS,                     # 记录 Harmony 校正的批次列和输出表示。
    'before_umap_basis': BEFORE_UMAP_BASIS, 'after_umap_basis': AFTER_UMAP_BASIS,               # 记录去批次前后两套 UMAP 坐标名称。
    'sample_key': SAMPLE_KEY, 'group_key': GROUP_KEY,                                           # 记录 sample 与预留 group 元数据列。
    'n_harmony_clusters': N_HARMONY_CLUSTERS,                                                   # 记录 Harmony 初始聚类数。
    'leiden_resolution': LEIDEN_RESOLUTION, 'seed': RANDOM_SEED,                                # 记录 Leiden 分辨率和随机种子。
}
metadata_columns = ['cohort', SAMPLE_KEY, 'leiden', 'cell_type']                                # 定义必须导出的样本、cluster 和注释列。
if GROUP_KEY in adata_work.obs:                                                                 # 只有真实 group 元数据存在时才加入导出列。
    metadata_columns.insert(2, GROUP_KEY)                                                       # 把 group 放在 sample 后，保持元数据顺序清晰。
current_metadata = adata_work.obs[metadata_columns].copy()                                      # 准备本次运行的逐细胞元数据，用于写出或一致性核验。
current_metadata.index.name = 'cell_id'                                                         # 明确细胞索引列名，保证导出结构稳定。
OVERWRITE_DATA_OUTPUTS = False                                                                  # 重画图片时默认保护已有 H5AD、注释表和参数文件。
data_outputs = {                                                                                # 集中声明需要保护的正式数据产物。
    'h5ad': OUTPUT_DIR / 'rna_e_cyl_zcp_confirmed.h5ad',                                        # 最终 AnnData 文件。
    'cell_metadata': OUTPUT_DIR / 'cell_id_cell_type.tsv',                                      # 逐细胞注释表。
    'parameters': OUTPUT_DIR / 'confirmed_parameters.json',                                     # 已确认参数。
}
existing_outputs = [name for name, path in data_outputs.items() if path.exists()]               # 检查已有正式数据产物。
if existing_outputs and not OVERWRITE_DATA_OUTPUTS:                                             # 默认不覆盖，避免单纯重画图片改动正式结果。
    if len(existing_outputs) != len(data_outputs):                                              # 部分文件存在意味着结果目录状态不完整。
        raise RuntimeError(f'Partial data outputs exist: {existing_outputs}; review before rerun')
    saved_metadata = pd.read_csv(data_outputs['cell_metadata'], sep='\t', index_col='cell_id', keep_default_na=False)
    saved_metadata.index = saved_metadata.index.astype(str)                                     # 统一索引类型后再逐项比较。
    current_compare = current_metadata.copy()                                                   # 不改变待导出的原始数据。
    current_compare.index = current_compare.index.astype(str)                                   # 统一当前索引类型。
    current_compare = current_compare.astype(str)                                               # categorical 等类型转字符串以稳定比较。
    saved_compare = saved_metadata[current_compare.columns].astype(str)                         # 按当前列顺序比较已有注释。
    if not current_compare.index.equals(saved_compare.index) or not current_compare.equals(saved_compare):
        raise RuntimeError('Current notebook annotations differ from existing outputs; set OVERWRITE_DATA_OUTPUTS=True only after review')
    print(f'Preserved matching existing data outputs: {list(data_outputs.values())}')           # 明确告知本次只更新图，不覆盖正式数据。
else:                                                                                           # 首次输出或人工明确允许覆盖时写出正式数据。
    adata_work.write_h5ad(data_outputs['h5ad'], compression='gzip')                             # 保存最终 H5AD。
    current_metadata.to_csv(data_outputs['cell_metadata'], sep='\t')                            # 保存逐细胞注释表。
    with data_outputs['parameters'].open('w') as handle:                                        # 保存独立参数 JSON。
        json.dump(adata_work.uns['notebook_parameters'], handle, indent=2, sort_keys=True)      # 便于阅读和版本比较。
with (OUTPUT_DIR / 'figure_manifest.json').open('w') as handle:                                 # 保存 notebook 图片的机器可读清单。
    json.dump({'figure_dir': str(FIGURE_DIR), 'n_figures': len(SAVED_FIGURES),                  # 记录图片目录和本次实际保存数量。
               'figures': SAVED_FIGURES}, handle, indent=2, sort_keys=True)                     # 记录每张图片的绝对路径。
print(f'Saved {len(SAVED_FIGURES)} figures to {FIGURE_DIR}')                                    # 在最终单元汇报图片输出数量。

: 